# The forward test: new days (added after looking)

Plan committed before these days existed: see [PLAN_clean_checks.md](PLAN_clean_checks.md).

- **Models:** step 9's 420 runs (4 sets close to close, 2 open to close, 7 models, 10 markets), trained on every day to 2026-10-23 and frozen. Their settings, fair bars and hashes are in `forward_test_models.csv`.
- **Days:** 2026-10-24 to 2027-11-26, which no choice in this project can have seen. The same days for every set.
- **Expected:** no edge from A, B or C; D beats A close to close for most models; no open-to-close gain in most markets.

`forward_test.py score` checks the hashes and makes the predictions. This notebook scores them with step 9's code, once.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from classifiers import overview
from data import MARKETS, NEW_MARKETS
from evaluation import bootstrap_range, holm, mcnemar
from forward_test import FOLDER
from models import pooled
from more_inputs import MODELS, RUNS

markets = MARKETS + NEW_MARKETS
preds = pd.read_pickle(FOLDER / "predictions.pkl")  # written by forward_test.py score

def together(target, s, model):
    """One model's predictions on every market, one row per market-day."""
    return pd.concat([preds[target, s, model, m].assign(market=m) for m in markets])

def right(p):
    return (p["p_up"] > 0.5) == p["up"]

pd.Series({m: len(preds["close to close", "A", "LDA", m]) for m in markets}, name="new days").to_frame().T

## Overview

Mean AUC over the market-years (10 markets; the end of 2026, and 2027), per model and set. A coin flip scores 0.50.

In [ ]:
pd.DataFrame({f"{s}, {target}": overview({model: {m: preds[target, s, model, m] for m in markets} for model in MODELS})["mean AUC"]
              for target, sets in RUNS.items() for s in sets}).round(3)

## Main result: better than A?

Each model with set B, C or D against the same model with set A, on the same days (all markets and years together). McNemar's test counts days where only one of the two was right. Holm adjusts over the 21 close-to-close comparisons. The AUC range is from whole weeks drawn at random.

In [ ]:
def against_a(target, s, model):
    a, b = together(target, "A", model), together(target, s, model)
    assert a.index.equals(b.index) and (a["market"] == b["market"]).all()  # the same days
    only_b, only_a, p = mcnemar(b["up"], b["p_up"] > 0.5, a["p_up"] > 0.5)
    daily = pd.DataFrame({"up": b["up"], "a": a["p_up"], "b": b["p_up"]})
    auc_gain = lambda f: roc_auc_score(f["up"], f["b"]) - roc_auc_score(f["up"], f["a"])
    low, high = bootstrap_range(daily, auc_gain)
    return {"accuracy gain, points": 100 * (right(b).mean() - right(a).mean()), "AUC gain": auc_gain(daily),
            "AUC gain, low": low, "AUC gain, high": high,
            "days only it was right": only_b, "days only A was right": only_a, "p": p}

main = pd.DataFrame({(s, model): against_a("close to close", s, model) for s in "BCD" for model in MODELS}).T
main["p (Holm)"] = holm(main["p"])
main.round(3)

Open to close (buy at the open, sell at the close): set D against A, Holm over these 7.

In [ ]:
tradeable = pd.DataFrame({model: against_a("open to close", "D", model) for model in MODELS}).T
tradeable["p (Holm)"] = holm(tradeable["p"])
tradeable.round(3)

## Any edge at all?

Pooled accuracy minus the fair bar, per model and market. Holm adjusts within each set (70 tests), as in step 8.

In [ ]:
edge = pd.DataFrame([{"target": target, "set": s, "model": model, "market": m,
                      "gain": pooled(preds[target, s, model, m])["accuracy %"] - pooled(preds[target, s, model, m])["fair bar %"],
                      "p": pooled(preds[target, s, model, m])["p vs fair"]}
                     for target, sets in RUNS.items() for s in sets for model in MODELS for m in markets])
edge["p (Holm)"] = edge.groupby(["target", "set"])["p"].transform(holm)
edge.groupby(["target", "set"]).agg(
    tests=("p", "size"), mean_gain=("gain", "mean"), above=("gain", lambda g: (g > 0).sum()),
    significant=("p", lambda p: (p < 0.05).sum()), significant_after_holm=("p (Holm)", lambda p: (p < 0.05).sum())).round(2)

Each set's gains over the fair bar, in points (model × market).

In [ ]:
for (target, s), table in edge.groupby(["target", "set"]):
    print(f"set {s}, {target}")
    display(table.pivot(index="model", columns="market", values="gain").round(1))

CONCLUSIONS: written after the one run.